# ⚡ Twitter Post Classifier - JEV / Laya (Colab GPU Edition)

Clasificación semántica ultrarrápida de posts de X (Twitter) impulsada por **JEV / TypeSafe System One (mmBERT)** y acelerada por GPU NVIDIA en Google Colab.

### ✨ Características del Entorno:
- 🚀 **Inferencia por GPU:** Aceleración por hardware CUDA (~5-20 ms por post vs ~300 ms en CPU).
- 🌐 **Selección de Túnel Web:** Elige entre **Cloudflare Tunnels**, **Colab Native URL (Local Directo)** o **Localtunnel** para abrir el Dashboard interactivo.
- 🔊 **Protocolo Anti-Desconexión:** Audio silencioso y pings para evitar que Colab suspenda la sesión.
- 💾 **Respaldo a Google Drive:** Exporta tu base de datos SQLite clasificada para no perder tus datos al apagar la instancia.

In [ ]:
#@title 1. Inicializar Entorno y Dependencias (GPU + UV)
#@markdown Clona el repositorio desde GitHub e instala dependencias a máxima velocidad con `uv`.

import os
import shutil
import subprocess
import sys
import torch
from IPython.display import HTML, display

display(HTML('''
<div style="background: linear-gradient(90deg, #1d9bf0 0%, #0d47a1 100%); color: white; padding: 14px; border-radius: 8px; font-weight: bold; text-align: center; font-family: sans-serif;">
  ⚡ Inicializando Entorno Twitter Classifier con Aceleración GPU...
</div>
'''))

# --- Parámetros de Configuración ---
REPO_URL = "https://github.com/cchrrisppatrickk/twitter_post_classifier_jev-laya.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
MOUNT_DRIVE = False #@param {type:"boolean"}
WORKSPACE = "/content/twitter_post_classifier"

# 1. Comprobación de Hardware GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU Detectada: \033[92m{gpu_name}\033[0m")
else:
    print("⚠️ \033[93mATENCIÓN: No se detectó GPU CUDA activa. Ve al menú superior 'Entorno de ejecución' > 'Cambiar tipo de entorno' y selecciona GPU T4.\033[0m")

# 2. Instalar herramientas de alta velocidad (UV y Cloudflared)
print("\n📦 [1/3] Instalando UV y Cloudflared...")
subprocess.run(["pip", "install", "-q", "uv"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# 3. Montar Google Drive si fue seleccionado
if MOUNT_DRIVE:
    print("\n💾 [2/3] Montando Google Drive para persistencia...")
    from google.colab import drive
    drive.mount('/content/drive')

# 4. Clonar o actualizar repositorio
print("\n📂 [3/3] Descargando código del proyecto...")
if not os.path.exists(WORKSPACE):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORKSPACE])
else:
    print("   🔄 Repositorio existente. Obteniendo últimos cambios (git pull)...")
    subprocess.run(["git", "pull"], cwd=WORKSPACE)

# Instalar dependencias con UV
print("\n🛠️ Instalando librerías Python vía UV...")
req_path = os.path.join(WORKSPACE, "backend", "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["uv", "pip", "install", "--system", "-r", req_path])
else:
    subprocess.run(["uv", "pip", "install", "--system", "laya", "fastapi", "uvicorn[standard]", "pydantic", "pydantic-settings", "psutil", "aiosqlite", "ijson", "python-multipart"])

print("\n✅ [SISTEMA LISTO] Entorno configurado exitosamente para inferencia.")

In [ ]:
#@title 2. Protocolo Anti-Desconexión (Keep-Alive)
#@markdown Mantiene la sesión activa en segundo plano mediante un bucle de audio silencioso y pulsos JavaScript.

from IPython.display import HTML, display

display(HTML('''
<div style="padding: 10px; background: #1a1a1a; border-left: 4px solid #1d9bf0; color: #eee; font-family: sans-serif; border-radius: 4px;">
  <b>🔊 Keep-Alive Activo</b><br>
  <span style="font-size: 12px; color: #aaa;">Previene la suspensión de la pestaña de Colab durante análisis prolongados.</span>
  <div style="margin-top: 8px;">
    <audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="height: 30px; width: 250px;"></audio>
  </div>
</div>
<script>
  function keepAlive() {
    console.log("[Keep-Alive] Pulso emitido: " + new Date().toISOString());
  }
  setInterval(keepAlive, 60000);
</script>
'''))

In [ ]:
#@title 3. Iniciar Dashboard de Clasificación (GPU + Túnel Web)
#@markdown Selecciona el proveedor de enlace para visualizar el Dashboard interactivo.

import os
import re
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from IPython.display import HTML, display

# --- Opciones de Ejecución ---
TUNNEL_PROVIDER = "Cloudflared (Recomendado - Sin Registro)" #@param ["Cloudflared (Recomendado - Sin Registro)", "Colab Native URL (Local Directo)", "Localtunnel"]
INFERENCE_DEVICE = "cuda" #@param ["cuda", "cpu"]
PORT = 8000
WORKSPACE = "/content/twitter_post_classifier"

display(HTML('''
<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: #000; padding: 12px; border-radius: 6px; font-weight: bold; text-align: center; font-family: sans-serif;">
  🚀 Arrancando Servidor FastAPI & Motor System 1...
</div>
'''))

# Configurar variables de entorno consumidas por config.py
os.environ["DEVICE"] = INFERENCE_DEVICE
os.environ["API_PORT"] = str(PORT)

def wait_for_port(port, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        res = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if res == 0:
            return True
    return False

def render_tunnel_banner(name, url, note=""):
    display(HTML(f'''
    <div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: #000; padding: 16px; border-radius: 8px; font-weight: bold; font-size: 16px; margin: 15px 0; text-align: center; box-shadow: 0 4px 6px rgba(0,0,0,0.15); font-family: sans-serif;">
      🚀 <b>{name} LISTO:</b> <a href="{url}" target="_blank" style="color: #002b49; text-decoration: underline; margin-left: 8px;">{url}</a>
      {f'<br><span style="font-size: 12px; color: #003b1f; margin-top: 4px; display: inline-block;">{note}</span>' if note else ''}
    </div>
    '''))

def run_tunnel(provider, port):
    if not wait_for_port(port):
        print("⚠️ Tiempo de espera agotado esperando al puerto del servidor.")
        return

    if "Cloudflared" in provider:
        p = subprocess.Popen(
            ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate", "--edge-ip-version", "4"],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        for line in p.stderr:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                render_tunnel_banner("CLOUDFLARE TURBO TUNNEL", match.group(0), "Acceso seguro HTTPS sin registro ni contraseñas.")
                break

    elif "Colab Native" in provider:
        try:
            from google.colab import output
            colab_url = output.eval_js(f'google.colab.kernel.proxyPort({port})')
            render_tunnel_banner("ENLACE DIRECTO COLAB (LOCAL)", colab_url, "Conexión directa interna mediante proxy de Google Colab.")
        except Exception as e:
            print(f"Error generando proxy nativo de Colab: {e}")

    elif "Localtunnel" in provider:
        if not os.path.exists("/usr/local/bin/lt"):
            subprocess.run("npm install -g localtunnel > /dev/null 2>&1", shell=True)
        try:
            ip = urllib.request.urlopen('https://ipv4.icanhazip.com', timeout=5).read().decode('utf8').strip()
        except:
            ip = "No disponible"
        print(f"\n👉 **Contraseña de Localtunnel (IP):** \033[93m{ip}\033[0m (Ingresa esta IP si te solicita password)\n")
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stdout:
            if "your url is" in line.lower():
                url = line.split("is:")[-1].strip() if "is:" in line else line.split("is")[-1].strip()
                render_tunnel_banner("LOCALTUNNEL", url, f"Contraseña de acceso (IP pública): {ip}")
                break

# Iniciar hilo en segundo plano para el túnel
threading.Thread(target=run_tunnel, daemon=True, args=(TUNNEL_PROVIDER, PORT)).start()

# Lanzar el servidor FastAPI con Uvicorn
if os.path.exists(os.path.join(WORKSPACE, "backend", "app", "main.py")):
    os.chdir(WORKSPACE)
    print(f"\n[SISTEMA] Iniciando Servidor [Dispositivo: {INFERENCE_DEVICE.upper()}] [Túnel: {TUNNEL_PROVIDER}]")
    print("[SISTEMA] Transmitiendo registros en tiempo real...\n" + "-"*65)
    
    cmd = [
        sys.executable, "-m", "uvicorn", "app.main:app",
        "--app-dir", "backend",
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--lifespan", "on"
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
else:
    print(f"❌ Error: No se encontró backend/app/main.py en {WORKSPACE}. Verifica haber ejecutado la celda 1.")

In [ ]:
#@title 4. Respaldo de Base de Datos SQLite a Google Drive
#@markdown Guarda la base de datos con tus posts clasificados en Google Drive para conservarla al reiniciar Colab.

import os
import shutil
from datetime import datetime

DRIVE_DEST = "/content/drive/MyDrive/Twitter_Classifier_Backups" #@param {type:"string"}
DB_FILE = "/content/twitter_post_classifier/backend/data/database.sqlite"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

if os.path.exists(DB_FILE):
    os.makedirs(DRIVE_DEST, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dest_path = os.path.join(DRIVE_DEST, f"database_{timestamp}.sqlite")
    shutil.copy2(DB_FILE, dest_path)
    shutil.copy2(DB_FILE, os.path.join(DRIVE_DEST, "database_latest.sqlite"))
    print(f"✅ Base de datos respaldada con éxito en Google Drive:\n   📁 {dest_path}")
else:
    print(f"⚠️ No se encontró la base de datos en {DB_FILE}")